In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!py -m pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic
!py -m pip install langchain-text-splitters
!py -m pip install chromadb
!py -m pip show langchain
!py -m pip install docx2txt
!py -m pip install sentence-transformers

In [19]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_percentage_error, mean_absolute_error
import numpy as np
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
import shap
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Optional
import re

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
import json

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [20]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:latest" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

### Read data tool Code base

In [47]:
HYPERPARAM_REFERENCE = """
VALID HYPERPARAMETERS PER MODEL:

svr:
  C:        float, 0.01-1000.0    
  epsilon:  float, 0.0001-1.0     
  gamma:    'scale' or 0.001-10.0 
  kernel:   'rbf' or 'linear'

knn:
  n_neighbors: int, 3-30         
  weights:     'uniform' or 'distance'
  metric:      'euclidean' or 'manhattan'

xgb:
  n_estimators:    int, 400-600  
  learning_rate:   float, 0.01-0.05 
  max_depth:       int, 2-4       
  subsample:       float, 0.5-1.0 
  colsample_bytree:float, 0.5-1.0  
  reg_lambda:      float, 0.1-10.0
  reg_alpha:       float, 0.0-1.0  

gbr:
  n_estimators:    int, 100-1000
  learning_rate:   float, 0.001-0.2
  max_depth:       int, 2-6
  subsample:       float, 0.5-1.0
  min_samples_leaf:int, 1-20     
  min_samples_split:int, 2-20
"""
PARAM_BOUNDS = {
    "n_estimators":      (50, 1000, int),
    "max_depth":         (1, 10, int),
    "learning_rate":     (0.001, 0.5, float),
    "subsample":         (0.1, 1.0, float),
    "colsample_bytree":  (0.1, 1.0, float),
    "reg_lambda":        (0.0, 20.0, float),
    "reg_alpha":         (0.0, 10.0, float),
    "min_child_weight":  (1, 20, int),
    "gamma":             (0.0, 5.0, float),
    "min_samples_leaf":  (1, 50, int),
    "min_samples_split": (2, 50, int),
    "C":                 (0.001, 1000.0, float),
    "epsilon":           (0.0001, 1.0, float),
    "n_neighbors":       (1, 50, int),
}

In [48]:
def clamp_params(params: dict) -> dict:
    """
    Clamp every numeric hyperparameter to its valid range.
    Prevents InvalidParameterError (e.g. subsample=1.05) regardless
    of what the LLM suggests.
    """
    cleaned = {}
    for k, v in params.items():
        if k in PARAM_BOUNDS:
            lo, hi, typ = PARAM_BOUNDS[k]
            if isinstance(v, (int, float)):
                v = max(lo, min(hi, v))
                v = typ(v)
            cleaned[k] = v
        else:
            cleaned[k] = v  # categorical params (kernel, weights, metric) pass through
    return cleaned

In [49]:
MODEL_REGISTRY = {
    "xgb":   lambda cfg: XGBRegressor(
                 n_estimators    = cfg.get("n_estimators",    500),
                 learning_rate   = cfg.get("learning_rate",   0.03),
                 max_depth       = cfg.get("max_depth",        3),
                 subsample       = cfg.get("subsample",        0.8),
                 colsample_bytree= cfg.get("colsample_bytree", 0.8),
                 reg_lambda      = cfg.get("reg_lambda",       1.0),
                 reg_alpha       = cfg.get("reg_alpha",        0.0),
                 min_child_weight= cfg.get("min_child_weight", 1),
                 objective="reg:squarederror", verbosity=0,
                 n_jobs=-1, random_state=42),
    "gbr":   lambda cfg: GradientBoostingRegressor(
                 n_estimators    = cfg.get("n_estimators",    300),
                 learning_rate   = cfg.get("learning_rate",   0.05),
                 max_depth       = cfg.get("max_depth",        3),
                 subsample       = cfg.get("subsample",        0.8),
                 min_samples_leaf= cfg.get("min_samples_leaf", 3),
                 min_samples_split=cfg.get("min_samples_split",2),
                 random_state=42),
    "svr":   lambda cfg: SVR(
                 C       = cfg.get("C",       10.0),
                 epsilon = cfg.get("epsilon",  0.01),
                 gamma   = cfg.get("gamma",   "scale"),
                 kernel  = cfg.get("kernel",  "rbf")),
    "bayes": lambda cfg: BayesianRidge(),
    "knn":   lambda cfg: KNeighborsRegressor(
                 n_neighbors = cfg.get("n_neighbors", 7),
                 weights     = cfg.get("weights",    "distance"),
                 metric      = cfg.get("metric",     "euclidean"),
                 n_jobs=-1),
}

In [50]:
@tool
def read_data_tool(csv_path: str, prompt: str) -> dict:
    """
    Read CSV, returns column names and prompt
    prompt describes what the user wants to predict
    """
    df = pd.read_csv(csv_path, nrows=3)
    cols = df.columns.tolist()
    return {"all_columns": cols,
            "prompt": prompt}

In [51]:

read_data_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a data scehma classification agent.
    Call read_data_tool with the CSV path AND the prompt from the input.
    
    After getting the result, classify each column into one of: 
    - target: the single most appropriate column to predict
    - feature: all other columns useful for prediction
    - date: date/time identifier column (not useful as features
    
    Return only a JSON dict:
    {{
     "target":"column name",
     "feature_columns": ["col1","col2", ...]
     "date_columns": ["col1", ...]
     }}
    
    """),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

read_data_agent = create_tool_calling_agent(
    llm,
    [read_data_tool],
    read_data_prompt
)

read_data_executor = AgentExecutor(
    agent=read_data_agent,
    tools=[read_data_tool],
    verbose=True,
    return_intermediate_steps=True
)

### Preprocess tool

In [52]:
@tool
def preprocess_tool(df_path: str, target_col: str,
                    feature_cols: list, date_cols: list) -> dict:
    """
    Preprocess dataset: build lag/rolling features, split, save CSVs.
    """
    output_dir = "D:/mycode/Fintech-agent/data/processed_data"
    os.makedirs(output_dir, exist_ok=True)
 
    df = pd.read_csv(df_path)
 
    df = df.drop(columns=date_cols, errors="ignore")
    df = df[[c for c in feature_cols + [target_col] if c in df.columns]]
    df = df.dropna().reset_index(drop=True)

    TEST_SIZE = 8
    train_df  = df.iloc[:-TEST_SIZE].copy()
    test_df   = df.iloc[-TEST_SIZE:].copy()
    for lag in [1, 2, 4]:
        train_df[f"{target_col}_lag{lag}"] = train_df[target_col].shift(lag)
        for col in feature_cols:
            if col in train_df.columns:
                train_df[f"{col}_lag{lag}"] = train_df[col].shift(lag)
 
    for w in [2, 4]:
        train_df[f"{target_col}_roll{w}"] = (
            train_df[target_col].shift(1).rolling(w).mean()
        )
 
    train_df = train_df.dropna().reset_index(drop=True)

    combined = pd.concat([train_df, test_df], ignore_index=True)
 
    for lag in [1, 2, 4]:
        combined[f"{target_col}_lag{lag}"] = combined[target_col].shift(lag)
        for col in feature_cols:
            if col in combined.columns:
                combined[f"{col}_lag{lag}"] = combined[col].shift(lag)
 
    for w in [2, 4]:
        combined[f"{target_col}_roll{w}"] = (
            combined[target_col].shift(1).rolling(w).mean()
        )
 
    n_train   = len(train_df)
    train_out = combined.iloc[:n_train].dropna().reset_index(drop=True)
    test_out  = combined.iloc[n_train:].reset_index(drop=True)
 
    X_train = train_out.drop(columns=[target_col])
    y_train = train_out[target_col]
 
    X_test  = test_out.drop(columns=[target_col])
    y_test  = test_out[target_col]
 
    paths = {
        "X_train":  os.path.join(output_dir, "X_train.csv"),
        "X_test":   os.path.join(output_dir, "X_test.csv"),
        "y_train":  os.path.join(output_dir, "y_train.csv"),
        "y_test":   os.path.join(output_dir, "y_test.csv"),
    }
    X_train.to_csv(paths["X_train"], index=False)
    X_test.to_csv( paths["X_test"],  index=False)
    y_train.to_csv(paths["y_train"], index=False)
    y_test.to_csv( paths["y_test"],  index=False)
 
    return paths

In [53]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a data preprocessing agent.
Call preprocess_tool with:
- df_path: the csv file path
- target_col: the target column
- feature_col: list of feature column names
- date_cols: list of date column names to drop

Return the file paths dictionary.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


preprocess_agent = create_tool_calling_agent(
    llm,
    [preprocess_tool],
    preprocess_prompt
)


preprocess_executor = AgentExecutor(
    agent=preprocess_agent,
    tools=[preprocess_tool],
    verbose=True,
    return_intermediate_steps=True
)

In [54]:
def escape(text: str) -> str:
        #Escape all curly braces except LangChain placeholders.
        return text.replace("{", "{{").replace("}", "}}")

### Forecasting tool and codebase

In [55]:
def forecast(data_paths, model_name, model_params):
    X_train = pd.read_csv(data_paths["X_train"])
    X_test  = pd.read_csv(data_paths["X_test"])
    y_train = pd.read_csv(data_paths["y_train"]).squeeze()
    y_test  = pd.read_csv(data_paths["y_test"]).squeeze()
 
    model = MODEL_REGISTRY[model_name](model_params)
    model.fit(X_train, y_train)
 
    y_train_pred = model.predict(X_train)
 
    X_test_recursive  = X_test.copy()
    predictions       = []
    prediction_history = list(y_train.values)  # real history for rolling computation
 
    for i in range(len(X_test)):
        row  = X_test_recursive.iloc[[i]].copy()
        pred = model.predict(row)[0]
        predictions.append(pred)
        prediction_history.append(pred)
 
        for lag in [1, 2, 4]:
            col = f"cpiret_lag{lag}"
            if col in X_test_recursive.columns:
                future_idx = i + lag
                if future_idx < len(X_test_recursive):
                    X_test_recursive.at[
                        X_test_recursive.index[future_idx], col] = pred
 
        for w in [2, 4]:
            col = f"cpiret_roll{w}"
            if col in X_test_recursive.columns:
                next_idx = i + 1
                if next_idx < len(X_test_recursive):
                    roll_val = np.mean(prediction_history[-w:])
                    X_test_recursive.at[
                        X_test_recursive.index[next_idx], col] = roll_val
 
    predictions = np.array(predictions)
    try:
        if isinstance(model, (XGBRegressor, GradientBoostingRegressor)):
            explainer   = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_train)
        else:
            explainer   = shap.KernelExplainer(
                model.predict, shap.sample(X_train, min(30, len(X_train))))
            shap_values = explainer.shap_values(X_train)
 
        shap_result = (
            pd.DataFrame({
                "feature":    X_train.columns.tolist(),
                "importance": np.abs(shap_values).mean(axis=0)
            })
            .sort_values("importance", ascending=False)
            .head(5)
            .reset_index(drop=True)
            .to_dict(orient="records")
        )
    except Exception as e:
        shap_result = [{"error": str(e)}]
 
    return {
        "model_name":   model_name,
        "model_params": model_params,
        "train_r2":     round(float(r2_score(y_train, y_train_pred)), 4),
        "test_r2":      round(float(r2_score(y_test,  predictions)),  4),
        "test_rmse":    round(float(np.sqrt(mean_squared_error(y_test, predictions))), 6),
        "test_mape":    round(float(mean_absolute_percentage_error(y_test, predictions) * 100), 4),
        "test_mae":     round(float(mean_absolute_error(y_test, predictions)), 4),
        "shap_result":  shap_result,
        "y_test":       y_test.tolist(),
        "y_test_pred":  predictions.tolist()
    }

In [56]:
@tool
def forecast_tool(data_paths: dict, model_name: str, model_params: dict) -> dict:
    """
    Train a selected regression model using CSV paths and return evaluation metrics.
    model_name must be one of: bayes, gbr, xgb, svr, knn
    model_params: Required dict of hyperparameters
    """
    return forecast(data_paths, model_name=model_name, model_params=model_params)

In [57]:
def extract_suggested_model(llm_output: str, default: str) -> str:
    fenced = re.findall(r'```(?:json)?\s*(\{.*?\})\s*```', llm_output, re.DOTALL)
    for block in reversed(fenced):
        try:
            parsed = json.loads(block)
            mname  = parsed.get("parameters", parsed).get("model_name")
            if mname:
                return mname
        except Exception:
            continue
    return default

In [58]:
def extract_suggested_params(llm_output: str) -> dict:
    """
    Find the LAST JSON-looking dict containing 'model_params' in the LLM's
    text output (covers cases where LLM writes suggestions in prose with
    ```...``` blocks instead of calling the tool).
    """
    # Try fenced code blocks first
    fenced = re.findall(r'```(?:json)?\s*(\{.*?\})\s*```', llm_output, re.DOTALL)
    candidates = fenced if fenced else re.findall(r'\{[^{}]*"model_params"[^{}]*\{[^{}]*\}[^{}]*\}', llm_output, re.DOTALL)
 
    for block in reversed(candidates):  # last suggestion = most refined
        try:
            parsed = json.loads(block)
            params = parsed.get("parameters", parsed).get("model_params", {})
            if params:
                return clamp_params(params)
        except Exception:
            continue
    return {}

### RAG 

In [59]:
def build_rag(docx_path: str):
    loader   = Docx2txtLoader(docx_path)
    docs     = loader.load()
 
    splitter = RecursiveCharacterTextSplitter(
        chunk_size    = 600,
        chunk_overlap = 80,
        separators    = ["\n\n", "\n", ". "]
    )
    chunks = splitter.split_documents(docs)
    print(f"  RAG: {len(chunks)} chunks loaded from knowledge base")
 
    embeddings = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = Chroma.from_documents(chunks, embeddings)
    retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
    return retriever

def retrieve_context(retriever, query: str) -> str:
    """
    Build a retrieval query from current metrics and return
    relevant chunks from the knowledge base as a single string.
    """
    docs    = retriever.invoke(query)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    return context



def build_rag_query(history: list) -> str:
    """
    Build a retrieval query from the latest iteration's metrics
    so the most relevant knowledge base chunks are retrieved.
    """
    if not history:
        return "starting config macro forecasting small dataset quarterly"
 
    latest  = history[-1]
    tr2     = latest.get("train_r2", 0) or 0
    te2     = latest.get("test_r2",  0) or 0
    gap     = tr2 - te2
    model   = latest.get("model_name", "")
 
    if te2 < 0:
        return f"negative R2 severe overfitting {model} fix"
    elif gap > 0.3:
        return f"overfitting {model} regularization max_depth quarterly macro"
    elif tr2 < 0.3:
        return f"underfitting {model} increase capacity n_estimators"
    else:
        return f"fine tuning {model} test_r2 {te2:.2f} macro"

In [60]:

def build_forecast_prompt(history: list, rag_context: str = "") -> ChatPromptTemplate:
    def escape(text):
        return str(text).replace("{", "{{").replace("}", "}}")
 
    history_section   = ""
    forbidden_configs = []
 
    if history:
        history_text = "\n\nPREVIOUS ITERATION RESULTS:\n"
        for i, h in enumerate(history):
            cfg = h.get("model_params", {})
            forbidden_configs.append(cfg)
            # shap_top = h.get("shap_result", [])[:3]
            history_text += (
                f"\nIteration {i+1}: model={escape(h['model_name'])} "
                f"train_r2={h['train_r2']} test_r2={h['test_r2']} test_RMSE={h['test_rmse']}"
                f"config={escape(json.dumps(cfg))}\n"
                # f"SHAP ranking: {json.dumps(h['shap_result'])}"
            )
 
        latest = history[-1]
        gap    = (latest.get("train_r2") or 0) - (latest.get("test_r2") or 0)
        te2    = latest.get("test_r2") or 0
 
        if te2 < 0:
            diag = "NEGATIVE test_r2 — switch model or use max_depth=2, reg_lambda=10"
        elif gap > 0.3:
            diag = f"OVERFITTING gap={gap:.2f} — increase regularization"
        else:
            diag = "Fine-tune — small adjustments"
 
        history_section += (
            f"\nDIAGNOSIS: {diag}\n"
            f"FORBIDDEN CONFIGS (never use these configs): {escape(json.dumps(forbidden_configs))}\n"
        )
 
    rag_section = ""
    if rag_context:
        rag_section = (
            "\n\nKNOWLEDGE BASE — use these rules to choose model and parameters:\n"
            + escape(rag_context)
            + "\n"
        )
 
    system_msg = (
        "You are a forecasting optimization agent.\n"
        "You MUST call forecast_tool with THREE arguments:\n"
        "  1. data_paths: dict with X_train, X_test, y_train, y_test\n"
        "  2. model_name: one of xgb, gbr, svr, bayes, knn\n"
        "  3. model_params: NON-EMPTY dict of hyperparameters\n\n"
        + rag_section
        + escape(history_section)
        + "\nCall forecast_tool NOW with appropriate model_params following the knowledge base rules."
    )
 
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])

### Optimization Loop

In [15]:
def run_optimization_loop(csv_path: str, max_iter: int, prompt: str, rag_docx_path: str = None):
    retriever = build_rag(rag_docx_path) if rag_docx_path else None

    read_result = read_data_executor.invoke({"input":csv_path, prompt: {prompt}})    
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', read_result["output"], re.DOTALL)
    if match:
        result_dict = json.loads(match.group(1))
        target = result_dict["target"]
        feature_columns = result_dict["feature_columns"]
        date_columns = result_dict["date_columns"]

        print("Target:", target)
        print("Features:", feature_columns)
        print("Date Columns:", date_columns)
    
    preprocess_input = {
        "df_path": csv_path,
        "target_col": target,
        "feature_cols": feature_columns,
        "date_cols": date_columns,
    }
    preprocess_result  = preprocess_executor.invoke({"input": preprocess_input})
    preprocessed_paths = preprocess_result["intermediate_steps"][-1][1]
    history     = []
    best_metrics= None
    best_test_r2= -np.inf

    for iteration in range(1, max_iter + 1):
        print(f"OPTIMIZATION ITERATION {iteration} / {max_iter}")
        rag_context = ""

        if retriever:
            query       = build_rag_query(history)
            rag_context = retrieve_context(retriever, query)
            print(f"  RAG query: '{query}'")
            print(f"  RAG retrieved {len(rag_context.split())} words of context")
        
        forecast_prompt = build_forecast_prompt(history, rag_context)
        forecast_agent  = create_tool_calling_agent(
            llm, [forecast_tool], forecast_prompt)
        forecast_executor = AgentExecutor(
            agent=forecast_agent,
            tools=[forecast_tool],
            verbose=True,
            handle_parsing_errors=True,
            return_intermediate_steps=True
        )
 
        forecast_result = forecast_executor.invoke({"input": preprocessed_paths})
        metrics = forecast_result["intermediate_steps"][-1][1]
 
        if not metrics:
            print(f"  WARNING: Could not parse metrics at iteration {iteration}, skipping.")
            continue
        
        current_cfg = metrics.get("model_params",{})
        old_configs = [h.get("model_params", {}) for h in history]
        if current_cfg in old_configs:
            llm_text = forecast_result.get("output","")
            suggested_cfg = extract_suggested_params(llm_text)
            if suggested_cfg:
                metrics = forecast(preprocessed_paths,
                                   metrics.get("model_name"),
                                   suggested_cfg)
                metrics["model_params"] = suggested_cfg
            else:
                perturbed = {k: (round(v * 1.5,4) if isinstance(v,float)
                            else max(1,int(v*1.5)))
                            for k,v in current_cfg.items()
                            if isinstance(v, (int, float))}
                metrics = forecast(preprocessed_paths,
                                   metrics.get("model_name"),
                                   perturbed)
                metrics["model_params"] = perturbed
        
        # Sort SHAP by value
        if "shap_importance" in metrics:
            metrics["shap_importance"] = dict(
                sorted(metrics["shap_importance"].items(), 
                       key=lambda x: x[1], reverse=True))
    
 
        print(f"\n  Results:")
        print(f"  Model:     {metrics.get('model_name')}")
        print(f"  Config:    {metrics.get('model_params', {})}")
        print(f"  Train R²:  {metrics.get('train_r2')}")
        print(f"  Test R²:   {metrics.get('test_r2')}")
        print(f"  Test RMSE: {metrics.get('test_rmse')}")
 
        history.append(metrics)
 
        test_r2 = metrics.get("test_r2", -np.inf)
        if test_r2 > best_test_r2:
            best_test_r2  = test_r2
            best_metrics  = metrics
            print(f"  New best test R²: {best_test_r2:.4f}")
        else:
            print(f"  No improvement. Best so far: {best_test_r2:.4f}")
 
        # Early stop if R² is excellent
        if best_test_r2 >= 0.95:
            print(f"\n  Stopping early — test R² {best_test_r2:.4f} >= 0.95")
            break
 
    return best_metrics, history         

In [ ]:
best_metrics, history = run_optimization_loop(
        csv_path = r"D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv",
        max_iter = 10,
        prompt = "predict inflation / CPI rate",
        rag_docx_path = r"D:/mycode/Fintech-agent\Agentic System/rag_knowledge_base.docx"
    )

In [51]:
test_results = []
for each in history:
    test_results.append(each["test_r2"])
print(test_results)

[-1.4151, 0.2084, -0.7871]


## Manual agent tests

### CPI

In [61]:
def build_forecast_prompt(history: list, rag_context: str = "") -> ChatPromptTemplate:
    def escape(text):
        return str(text).replace("{", "{{").replace("}", "}}")
 
    history_section   = ""
    forbidden_configs = []
 
    if history:
        history_section = "\n\nPREVIOUS ITERATION RESULTS:\n"
        for i, h in enumerate(history):
            cfg = h.get("model_params", {})
            forbidden_configs.append({"model_name": h.get("model_name"), "model_params": cfg})
            history_section += (
                f"\nIteration {i+1}: model={escape(h.get('model_name',''))} "
                f"train_r2={h.get('train_r2')} test_r2={h.get('test_r2')} "
                f"test_rmse={h.get('test_rmse')} "
                f"config={escape(json.dumps(cfg))}\n"
            )
 
        latest = history[-1]
        gap    = (latest.get("train_r2") or 0) - (latest.get("test_r2") or 0)
        te2    = latest.get("test_r2") or 0
 
        if te2 < 0:
            diag = "NEGATIVE test_r2 — switch model entirely OR use max_depth=2, reg_lambda=10, min_child_weight=8"
        elif gap > 0.3:
            diag = f"OVERFITTING gap={gap:.2f} — increase reg_lambda, reduce max_depth, increase min_child_weight/min_samples_leaf"
        elif te2 < 0.3:
            diag = "LOW test_r2 — try a different model or reduce regularization"
        else:
            diag = "Reasonable performance — make small targeted adjustments"
 
        # Pull last suggested config from previous iteration's free text
        last_suggestion = history[-1].get("next_suggested_config", {})
        suggestion_text = ""
        if last_suggestion.get("model_params"):
            suggestion_text = (
                f"\nSUGGESTION CONFIGS TO USE:\n"
                f"model_name: {last_suggestion.get('model_name')}\n"
                f"model_params: {escape(json.dumps(last_suggestion.get('model_params')))}\n"
            )
 
        history_section += (
            f"\nDIAGNOSIS: {diag}\n"
            f"FORBIDDEN — do NOT use any of these exact (model_name, model_params) combinations:\n"
            f"{escape(json.dumps(forbidden_configs))}\n"
            f"{suggestion_text}"
        )
 
    rag_section = ""
    if rag_context:
        rag_section = (
            "\n\nKNOWLEDGE BASE — use these rules to choose model and parameters:\n"
            + escape(rag_context)
            + "\n"
        )
 
    valid_ranges = (
        "\nVALID PARAMETER RANGES (do not exceed these):\n"
        "  n_estimators: 50-1000 (int)\n"
        "  max_depth: 1-10 (int)\n"
        "  learning_rate: 0.001-0.5 (float)\n"
        "  subsample: 0.1-1.0 (float, NEVER above 1.0)\n"
        "  colsample_bytree: 0.1-1.0 (float, NEVER above 1.0)\n"
        "  reg_lambda: 0.0-20.0 (float)\n"
        "  reg_alpha: 0.0-10.0 (float)\n"
        "  min_child_weight: 1-20 (int)\n"
        "  gamma: 0.0-5.0 (float)\n"
        "  min_samples_leaf: 1-50 (int)\n"
        "  min_samples_split: 2-50 (int)\n"
        "  C: 0.001-1000.0 (float)\n"
        "  epsilon: 0.0001-1.0 (float)\n"
        "  n_neighbors: 1-50 (int)\n"
    )
 
    output_format = (
        "\nRESPONSE FORMAT — REQUIRED:\n"
        "1. FIRST, call forecast_tool immediately with your chosen model_name "
        "and model_params (must be NON-EMPTY, at least 3 keys, all values "
        "within VALID PARAMETER RANGES above).\n"
        "2. AFTER the tool returns results, in your text response propose "
        "the NEXT iteration's model_name and model_params as a fenced JSON "
        "block in EXACTLY this format:\n"
        "```json\n"
         + escape('{"name": "forecast_tool", "parameters": '
                 '{"model_name": "xgb", "model_params": {"max_depth": 2, '
                 '"reg_lambda": 5.0, "n_estimators": 200}}}') + "\n"
        "```\n"
        "Only include ONE such fenced block — your single best next suggestion, "
        "with values inside VALID PARAMETER RANGES.\n"
    )
 
    system_msg = (
        "You are a forecasting optimization agent.\n"
        "You MUST call forecast_tool with THREE arguments:\n"
        "  1. data_paths: dict with X_train, X_test, y_train, y_test\n"
        "  2. model_name: one of xgb, gbr, svr, bayes, knn\n"
        "  3. model_params: NON-EMPTY dict of hyperparameters (3+ keys)\n"
        # + valid_ranges
        + rag_section
        + escape(history_section)
        # + output_format
        + "\nCall forecast_tool NOW with appropriate model_params different from FORBIDDEN list."
    )
 
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])
 
 

In [62]:
def run_optimization_loop(csv_path: str, max_iter: int, prompt: str,
                          rag_docx_path: str = None):
    retriever = build_rag(rag_docx_path) if rag_docx_path else None
 
    preprocessed_paths = {
        "X_train": "D:/mycode/Fintech-agent/data/processed_data/X_train.csv",
        "X_test":  "D:/mycode/Fintech-agent/data/processed_data/X_test.csv",
        "y_train": "D:/mycode/Fintech-agent/data/processed_data/y_train.csv",
        "y_test":  "D:/mycode/Fintech-agent/data/processed_data/y_test.csv"
    }
    history      = []
    best_metrics = None
    best_test_r2 = -np.inf
 
    for iteration in range(1, max_iter + 1):
        print(f"\n{'='*65}\nOPTIMIZATION ITERATION {iteration} / {max_iter}")
 
        rag_context = ""
        if retriever:
            query       = build_rag_query(history)
            rag_context = retrieve_context(retriever, query)
            print(f"  RAG query: '{query}'")
            print(f"  RAG retrieved {len(rag_context.split())} words of context")
 
        forecast_prompt   = build_forecast_prompt(history, rag_context)
        forecast_agent    = create_tool_calling_agent(llm, [forecast_tool], forecast_prompt)
        forecast_executor = AgentExecutor(
            agent=forecast_agent, tools=[forecast_tool],
            verbose=True, handle_parsing_errors=True,
            return_intermediate_steps=True
        )
 
        forecast_result = forecast_executor.invoke({"input": preprocessed_paths})
        metrics = forecast_result["intermediate_steps"][-1][1]
 
        if not metrics:
            print(f"  WARNING: No metrics at iteration {iteration}, skipping.")
            continue
        if isinstance(metrics, str):
            try:
                metrics = json.loads(metrics)
            except Exception:
                continue
 
        #  Validate config wasn't reused; if it was, apply LLM's own suggestion 
        current_cfg = (metrics.get("model_name"), json.dumps(metrics.get("model_params", {}), sort_keys=True))
        old_configs = [
            (h.get("model_name"), json.dumps(h.get("model_params", {}), sort_keys=True))
            for h in history
        ]
 
        if current_cfg in old_configs:
            llm_text       = forecast_result.get("output", "")
            suggested_cfg  = extract_suggested_params(llm_text)
            suggested_name = extract_suggested_model(llm_text, metrics.get("model_name"))
 
            if suggested_cfg:
                print(f"  Applying suggested: model={suggested_name} params={suggested_cfg}")
                metrics = forecast(preprocessed_paths, suggested_name, suggested_cfg)
                metrics["model_name"]   = suggested_name
                metrics["model_params"] = suggested_cfg
            else:
                current_params = metrics.get("model_params", {})
                perturbed = {}
                for k, v in current_params.items():
                    if isinstance(v, float):
                        perturbed[k] = round(v * 1.5, 4)
                    elif isinstance(v, int):
                        perturbed[k] = max(1, int(v * 1.5))
                    else:
                        perturbed[k] = v
                perturbed = clamp_params(perturbed)
                print(f"  No suggestion found — perturbing: {perturbed}")
                metrics = forecast(preprocessed_paths, metrics.get("model_name"), perturbed)
                metrics["model_params"] = perturbed
        else:
            metrics["model_params"] = clamp_params(metrics.get("model_params", {}))
 
        llm_text = forecast_result.get("output", "")
        next_cfg = extract_suggested_params(llm_text)
        next_name = extract_suggested_model(llm_text, metrics.get("model_name"))
        if next_cfg:
            metrics["next_suggested_config"] = {
                "model_name":   next_name,
                "model_params": next_cfg
            }
 
        if "shap_importance" in metrics:
            metrics["shap_importance"] = dict(
                sorted(metrics["shap_importance"].items(),
                       key=lambda x: x[1], reverse=True))
 
        print(f"\n  Results:")
        print(f"  Model:     {metrics.get('model_name')}")
        print(f"  Config:    {metrics.get('model_params', {})}")
        print(f"  Train R²:  {metrics.get('train_r2')}")
        print(f"  Test R²:   {metrics.get('test_r2')}")
        print(f"  Test RMSE: {metrics.get('test_rmse')}")
        if metrics.get("next_suggested_config", {}).get("model_params"):
            print(f"  Next suggestion: {metrics['next_suggested_config']}")
 
        history.append(metrics)
 
        test_r2 = float(metrics.get("test_r2", -np.inf))
        if test_r2 > best_test_r2:
            best_test_r2 = test_r2
            best_metrics = metrics
            print(f"  New best test R²: {best_test_r2:.4f}")
        else:
            print(f"  No improvement. Best so far: {best_test_r2:.4f}")
 
        if best_test_r2 >= 0.95:
            print(f"\n  Stopping early — test R² {best_test_r2:.4f} >= 0.95")
            break
 
    return best_metrics, history


In [ ]:
best_metrics, history = run_optimization_loop(
        csv_path = r"D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv",
        max_iter = 10,
        prompt = "predict inflation / CPI rate",
        rag_docx_path = r"D:/mycode/Fintech-agent\Agentic System/rag_knowledge_base.docx"
    )

  RAG: 22 chunks loaded from knowledge base


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


OPTIMIZATION ITERATION 1 / 10
  RAG query: 'starting config macro forecasting small dataset quarterly'
  RAG retrieved 331 words of context


> Entering new AgentExecutor chain...

Invoking: `forecast_tool` with `{'data_paths': {'X_test': 'D:/mycode/Fintech-agent/data/processed_data/X_test.csv', 'X_train': 'D:/mycode/Fintech-agent/data/processed_data/X_train.csv', 'y_test': 'D:/mycode/Fintech-agent/data/processed_data/y_test.csv', 'y_train': 'D:/mycode/Fintech-agent/data/processed_data/y_train.csv'}, 'model_name': 'xgb', 'model_params': {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}}`


{'model_name': 'xgb', 'model_params': {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}, 'train_r2': 0.9973, 'test_r2': -0.4352, 'test_rmse': 0.00843, 'test_mape': 250.8352, 'test_mae': 0.0076, 'shap_result': [{'feature': 'cpiret_lag4', 'importance': 0.00319996802136302}, {'feature': 'cpiret_roll4', 'importance': 0.0011483654379844666}, {'feature': 't30ret', 'importance': 0.00097

### GDP

In [41]:

def run_optimization_loop(csv_path: str, max_iter: int):
    preprocess_input = {
        "df_path": csv_path,
        "target_col": 'gdp_growth',
        "feature_cols": [
        "b30ret","b20ret","b10ret","b7ret","b5ret","b2ret","b1ret",
        "t90ret","t30ret","cpiret","wti_price","fedfunds","nfci","anfci",
        "nfci_risk","nfci_credit","nfci_leverage","nfci_nonfinancial_leverage",
        "ppi","trade_balance","unrate","usd_index"],
        "date_cols":  ['date'],
    }
    
    preprocessed_paths = {'X_train': 'D:/mycode/Fintech-agent/data/processed_data/X_train.csv', 
                          'X_test': 'D:/mycode/Fintech-agent/data/processed_data/X_test.csv', 
                          'y_train': 'D:/mycode/Fintech-agent/data/processed_data/y_train.csv', 
                          'y_test': 'D:/mycode/Fintech-agent/data/processed_data/y_test.csv'}
    history     = []
    best_metrics= None
    best_test_r2= -np.inf

    for iteration in range(1, max_iter + 1):
        print(f"OPTIMIZATION ITERATION {iteration} / {max_iter}")
        
        forecast_prompt = build_forecast_prompt(history)
        forecast_agent  = create_tool_calling_agent(
            llm, [forecast_tool], forecast_prompt)
        
        forecast_executor = AgentExecutor(
            agent=forecast_agent,
            tools=[forecast_tool],
            verbose=True,
            handle_parsing_errors=True,
            return_intermediate_steps=True
        )
 
        forecast_result = forecast_executor.invoke(
            {"input": preprocessed_paths}
        )
        metrics = forecast_result["intermediate_steps"][-1][1]
 
        if not metrics:
            print(f"  WARNING: Could not parse metrics at iteration {iteration}, skipping.")
            continue
        
        current_cfg = metrics.get("model_params",{})
        old_configs = [h.get("model_params", {}) for h in history]
        if current_cfg in old_configs:
            llm_text = forecast_result.get("output","")
            suggested_cfg = extract_suggested_params(llm_text)
            if suggested_cfg:
                metrics = forecast(preprocessed_paths,
                                   metrics.get("model_name"),
                                   suggested_cfg)
                metrics["model_params"] = suggested_cfg
            else:
                perturbed = {k: (round(v * 1.5,4) if isinstance(v,float)
                            else max(1,int(v*1.5)))
                            for k,v in current_cfg.items()
                            if isinstance(v, (int, float))}
                metrics = forecast(preprocessed_paths,
                                   metrics.get("model_name"),
                                   perturbed)
                metrics["model_params"] = perturbed
        
        # Sort SHAP by value
        if "shap_importance" in metrics:
            metrics["shap_importance"] = dict(
                sorted(metrics["shap_importance"].items(), 
                       key=lambda x: x[1], reverse=True))
    
 
        print(f"\n  Results:")
        print(f"  Model:     {metrics.get('model_name')}")
        print(f"  Config:    {metrics.get('model_params', {})}")
        print(f"  Train R²:  {metrics.get('train_r2')}")
        print(f"  Test R²:   {metrics.get('test_r2')}")
        print(f"  Test RMSE: {metrics.get('test_rmse')}")
 
        history.append(metrics)
 
        test_r2 = metrics.get("test_r2", -np.inf)
        if test_r2 > best_test_r2:
            best_test_r2  = test_r2
            best_metrics  = metrics
            print(f"  New best test R²: {best_test_r2:.4f}")
        else:
            print(f"  No improvement. Best so far: {best_test_r2:.4f}")
 
        if best_test_r2 >= 0.95:
            print(f"\n  Stopping early — test R² {best_test_r2:.4f} >= 0.95")
            break
 
    return best_metrics, history         

In [ ]:
best_metrics, history = run_optimization_loop(
        csv_path = r"D:\mycode\Fintech-agent\data\macro\combined_quatarly_macro_data.csv",
        max_iter = 10
    )

## Report generation Agent

In [58]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a machine learning report generator.

Given evaluation metrics, generate a professional Markdown report including:

- Model Overview
- Performance Table
- Interpretation of metrics
- Overfitting Assessment
- Conclusion
"""
    ),
    ("human", "{input}")
])

report_chain = report_prompt | llm

In [59]:
if "shap_importance" in metrics:
    metrics["shap_importance"] = dict(
        sorted(metrics["shap_importance"].items(), key=lambda x: x[1], reverse=True))
    

# report_input = {
#     "history":      history,
#     "best_metrics": best_metrics,
#     "iterations":   len(history)
# }
# Step 3 — Generate report
report = report_chain.invoke(
    {"input": metrics} #best_metrics/report_input/metrics
)

print(report)

content="**Model Overview**\n================\n\nThe model used for this evaluation is a Gradient Boosting Regressor (GBR) with default configuration.\n\n**Performance Table**\n-------------------\n\n| Metric | Train | Test |\n| --- | --- | --- |\n| R2 Score | 0.955 | -0.264 |\n| RMSE | N/A | 0.0079 |\n| MAPE | N/A | 203.54 |\n\nNote: The test R2 score is negative, indicating that the model has overfit to the training data.\n\n**Interpretation of Metrics**\n---------------------------\n\n*   **R2 Score**: Measures the proportion of variance in the target variable explained by the model. A higher value indicates better fit.\n    *   Train R2 score: 0.955 (very good)\n    *   Test R2 score: -0.264 (poor, indicating overfitting)\n*   **RMSE (Root Mean Squared Error)**: Measures the average magnitude of the errors made by the model. A lower value indicates better performance.\n    *   Test RMSE: 0.0079\n*   **MAPE (Mean Absolute Percentage Error)**: Measures the average absolute percentage

In [5]:
with open("result/forecast_report.md", "w", encoding="utf-8") as f:
    f.write(report["content"])

print("Report saved to forecast_report.md")


Report saved to forecast_report.md


### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 
